# Python 函数式编程完全指南

> **阅读方式**：按顺序执行每个代码单元，观察输出，再对照上方原理说明。所有示例均已真实执行、零报错，输出已嵌入 Notebook。
> **教学假设的业务场景**（原文占位符未填写，按本目录上下文合理假设）：
> **LLMOps 平台的 API 服务开发** —— 处理 LLM 请求的 Prompt 清洗管道、流式响应转换、结果缓存（记忆化）、失败重试与日志中间件、不可变配置更新。
> 全文示例尽量围绕这一场景取材，而不是空洞 demo。

---

## 学习目标

学完本 Notebook，你应该能够：

1. 说清楚**一等函数、纯函数、不可变性、高阶函数、闭包**这五个核心概念，并用代码验证它们；
2. 熟练使用 `map / filter / reduce / lambda`，并知道什么时候**不该**用它们（改用推导式）；
3. 掌握 `functools`（`partial / lru_cache / cache / singledispatch / wraps / total_ordering`）与 `itertools` 的常用工具；
4. 会写**函数组合 compose / pipe、柯里化、生成器管道**，把多个小函数串成一条数据流水线；
5. 识别并修复 10 个函数式编程常见陷阱（延迟绑定、可变默认参数、迭代器耗尽等）。

## 一句话本质

> **函数式编程把计算看作「函数的求值」：用纯函数 + 不可变数据 + 函数组合来表达逻辑，
> 而不是用「修改状态 + 循环赋值」来表达。**
> Python 不是纯函数式语言，但推导式、生成器、functools、itertools 是它的四大支柱。

---
# 〇、前置知识：先补齐地基，再盖楼

**本章解决什么问题**：函数式编程建立在 4 个语言基础之上，先逐个验证清楚，后面正文才不会卡壳。

| 前置知识 | 为什么需要 | 验证方式 |
|---|---|---|
| ① 函数是对象 | 一等函数、高阶函数、装饰器的前提 | 观察函数的 `type`、`id`、属性 |
| ② 可变 vs 不可变 | 纯函数 / 不可变风格的核心区分 | 对比 `list` 与 `tuple` 修改时的行为 |
| ③ 迭代器协议 | `map/filter/itertools` 全部返回迭代器 | 用 `iter()/next()` 手动驱动一次 |
| ④ 生成器入门 | 惰性求值、管道、无限序列的基础 | 用计数器证明生成器「按需计算」 |
| ⑤ 作用域与 nonlocal | 闭包读写外层变量的语法基础 | 闭包内 `nonlocal` 计数 |

下面用 3 个代码单元逐一验证。

In [1]:
# ---------- 前置验证 ①：函数是对象 ----------
def hello():
    return "hi"

# 1) 函数本身是 function 类型的对象
print("① type(hello) =", type(hello).__name__)
# 2) 有自己的身份（内存地址）与属性
print("② id(hello)   =", id(hello))
print("③ __name__    =", hello.__name__)
# 4) 可以塞进字典 —— 后面注册表 / 事件处理都靠这一点
registry = {"greet": hello}
print("④ 存入字典后取出来调用:", registry["greet"]())

# ---------- 前置验证 ②：可变 vs 不可变 ----------
lst = [1, 2]
tup = (1, 2)
print("⑤ 修改 list 前后 id 是否不变（原地修改）:",
      id(lst), "->", (lst.append(3), id(lst))[1], "id 相同 =", id(lst) == id(lst))
# tuple 修改会直接报错，用 try/except 捕获演示（错误写法见下方反例章节）
try:
    tup[0] = 99          # 错误写法：tuple 不支持原地修改
except TypeError as e:
    print("⑥ tuple 原地修改触发报错:", e)

① type(hello) = function
② id(hello)   = 4430985504
③ __name__    = hello
④ 存入字典后取出来调用: hi
⑤ 修改 list 前后 id 是否不变（原地修改）: 4425516928 -> 4425516928 id 相同 = True
⑥ tuple 原地修改触发报错: 'tuple' object does not support item assignment


In [2]:
# ---------- 前置验证 ③：迭代器协议 ----------
nums = [10, 20, 30]
it = iter(nums)          # 可迭代对象 -> 迭代器（实现了 __next__）
print("① iter() 返回:", type(it).__name__)
print("② next(it) 第 1 次:", next(it))
print("③ next(it) 第 2 次:", next(it))
print("④ next(it) 第 3 次:", next(it))
try:
    next(it)             # 耗尽后再取会触发 StopIteration（循环底层就靠它终止）
except StopIteration as e:
    print("⑤ 耗尽后 next() 触发:", type(e).__name__)

# ---------- 前置验证 ④：生成器 = 惰性迭代器 ----------
steps = []

def gen():
    for i in range(3):
        steps.append(f"产生了 {i}")   # 副作用仅用于观察求值时机
        yield i

g = gen()
print("⑥ 创建生成器后立即观察 steps =", steps, "（还没有任何计算）")
next(g); print("⑦ 取 1 个之后的 steps =", steps)
next(g); print("⑧ 取 2 个之后的 steps =", steps)
print("⑨ 结论：生成器确实按需计算（惰性），这是函数式管道处理大数据的基础")

① iter() 返回: list_iterator
② next(it) 第 1 次: 10
③ next(it) 第 2 次: 20
④ next(it) 第 3 次: 30
⑤ 耗尽后 next() 触发: StopIteration
⑥ 创建生成器后立即观察 steps = [] （还没有任何计算）
⑦ 取 1 个之后的 steps = ['产生了 0']
⑧ 取 2 个之后的 steps = ['产生了 0', '产生了 1']
⑨ 结论：生成器确实按需计算（惰性），这是函数式管道处理大数据的基础


In [3]:
# ---------- 前置验证 ⑤：作用域与 nonlocal ----------
def make_counter_demo():
    count = 0                # 外层函数的局部变量
    def inc():
        nonlocal count       # 闭包内声明：写外层变量必须用 nonlocal
        count += 1
        return count
    return inc

c = make_counter_demo()
print("① 闭包计数器:", c(), c(), c())
print("② 外层变量被闭包记住的证据 -> __closure__:", c.__closure__[0].cell_contents)

# 反面对照：不加 nonlocal 直接赋值会怎样？（错误写法，只展示原理不实际运行）
demo = """
def make_bad():
    count = 0
    def inc():
        count += 1   # UnboundLocalError：count 被当成 inc 的局部变量
        return count
    return inc
"""
try:
    exec(demo)  # 只验证语法可解析，真正调用错误的写法放到后面反例章节
    print("③ 反面示例语法合法（运行时报错行为在反例章节验证）")
except SyntaxError:
    print("③ 反面示例存在语法错误")

① 闭包计数器: 1 2 3
② 外层变量被闭包记住的证据 -> __closure__: 3
③ 反面示例语法合法（运行时报错行为在反例章节验证）


---
# 一、函数是一等对象

**本章解决什么问题**：整个函数式编程的地基 —— 函数能不能像 `int`、`str` 一样被赋值、传参、返回、存容器？

**要点**

- 一等对象 = 可以**赋值给变量、作为参数传递、作为返回值、存入数据结构**的对象；
- Python 中函数与 `1`、`"abc"` 地位相同，都是堆上的对象；
- 这是高阶函数（第 4 章）、闭包（第 6 章）、装饰器（第 7/13 章）的共同前提。

| 能力 | 语法 | 典型用途 |
|---|---|---|
| 赋值 | `f = greet` | 别名、回调变量 |
| 传参 | `apply(greet, x)` | 回调、策略注入 |
| 返回 | `return greeter` | 工厂函数、装饰器 |
| 存容器 | `[str.upper, str.lower]` | 函数注册表、事件分发 |

In [4]:
# ---------- 第 1 章：函数是一等对象的四种玩法 ----------
def greet(name):
    """返回一句问候。"""
    return f"Hello, {name}"

# 1) 赋值给变量：注意没有括号 —— 传的是函数对象本身，不是调用结果
f = greet
print("① 赋值给变量:", f("Alice"))          # Hello, Alice

# 2) 作为参数传递（高阶函数的雏形）
def apply(func, value):
    return func(value)
print("② 作为参数:", apply(greet, "Bob"))   # Hello, Bob

# 3) 作为返回值（工厂函数：返回一个「记住」了 prefix 的新函数）
def make_greeter(prefix):
    def greeter(name):
        return f"{prefix}, {name}"
    return greeter

hi = make_greeter("Hi")
print("③ 工厂函数:", hi("Carol"))            # Hi, Carol

# 4) 存入数据结构（LLMOps 场景：把多个文本规范化函数注册成管线）
funcs = [str.upper, str.lower, str.title]
print("④ 存入列表批量执行:", [f("hello") for f in funcs])
# ['HELLO', 'hello', 'Hello']

# 5) 验证：赋值不产生新函数，只是多了一个名字指向同一个对象
print("⑤ f is greet =", f is greet, "| id 相同 =", id(f) == id(greet))

① 赋值给变量: Hello, Alice
② 作为参数: Hello, Bob
③ 工厂函数: Hi, Carol
④ 存入列表批量执行: ['HELLO', 'hello', 'Hello']
⑤ f is greet = True | id 相同 = True


---
# 二、纯函数

**本章解决什么问题**：什么样的函数容易测试、能缓存、能并行？—— 纯函数。

**判定标准（两条同时满足）**

1. 相同输入**永远**返回相同输出；
2. 没有副作用：不修改外部状态、不打印、不写文件、不修改传入参数。

**纯函数的四个好处**

| 好处 | 原因 | 对应工具 |
|---|---|---|
| 易于测试 | 不需要 mock 外部状态 | 单元测试 |
| 易于推理 | 只看签名和输入即可推导结果 | 类型注解 |
| 可缓存 | 相同输入必然相同输出 | `lru_cache` |
| 可并行 | 无共享状态，天然线程安全 | 多进程 |

In [5]:
# ---------- 第 2 章：纯函数 vs 非纯函数 ----------
# 1) 纯函数：只依赖参数，只通过返回值输出
def add(a, b):
    return a + b
print("① 纯函数 add(1, 2) =", add(1, 2))

# 2) 非纯函数 A：读写全局状态（隐藏依赖）
total = 0
def add_to_total(x):
    global total        # 修改外部状态 -> 不纯
    total += x
    return total
print("② 非纯函数连续调用两次:", add_to_total(1), add_to_total(2), "| total =", total)

# 3) 非纯函数 B：修改传入的可变对象（副作用泄漏到调用方）
def append_item(lst, item):
    lst.append(item)    # 改的是调用方的列表 -> 不纯
    return lst

# 4) 纯函数版：返回新列表，不碰原数据
def pure_append(lst, item):
    return lst + [item]

original = [1, 2]
new = pure_append(original, 3)
print("③ 纯函数: 原列表 =", original, "| 新列表 =", new)
# original 保持 [1, 2]，new 是 [1, 2, 3]

# 5) 对照：非纯版本会「污染」调用方数据
victim = [1, 2]
append_item(victim, 3)
print("④ 非纯函数: 调用方列表被改成 =", victim)   # [1, 2, 3] —— 隐蔽 bug 来源

① 纯函数 add(1, 2) = 3
② 非纯函数连续调用两次: 1 3 | total = 3
③ 纯函数: 原列表 = [1, 2] | 新列表 = [1, 2, 3]
④ 非纯函数: 调用方列表被改成 = [1, 2, 3]


---
# 三、不可变性

**本章解决什么问题**：函数式代码倾向「不修改、只新建」，怎么做才地道？

**三种层面的不可变写法**

| 场景 | 可变写法（改原对象） | 不可变写法（建新对象） |
|---|---|---|
| 列表 | `lst.append(4)` | `new = lst + [4]` |
| 字典 | `d["b"] = 2` | `new_d = {**d, "b": 2}` |
| 自定义对象 | 普通类随意改属性 | `@dataclass(frozen=True)` |

**Python 内置不可变类型**：`int`、`float`、`str`、`tuple`、`frozenset`、`bytes`。

In [6]:
# ---------- 第 3 章：不可变风格 ----------
# 1) 列表：可变写法 vs 不可变写法
lst = [1, 2, 3]
lst.append(4)                     # 可变：原地修改，id 不变
print("① 可变写法:", lst)
lst2 = [1, 2, 3]
new_lst = lst2 + [4]               # 不可变：产生新列表
print("② 不可变写法: 原 =", lst2, "| 新 =", new_lst)

# 2) 字典：合并时用 ** 展开，而不是原地改
d = {"a": 1}
new_d = {**d, "b": 2}
print("③ 不可变字典合并: 原 =", d, "| 新 =", new_d)

# 3) 自定义类：frozen dataclass 从语言层面禁止修改
from dataclasses import dataclass

@dataclass(frozen=True)
class ModelConfig:
    name: str
    temperature: float

cfg = ModelConfig("gpt", 0.7)
print("④ frozen 实例:", cfg)

# 4) 反例（错误写法）：试图修改 frozen 实例会抛 FrozenInstanceError
try:
    cfg.temperature = 1.5          # 错误写法：frozen 实例禁止属性赋值
except Exception as e:
    print("⑤ 触发报错:", type(e).__name__, "-", e)

# 5) 正确写法：用 dataclasses.replace 生成新实例
from dataclasses import replace
cfg2 = replace(cfg, temperature=1.5)
print("⑥ replace 生成新实例: 原 =", cfg.temperature, "| 新 =", cfg2.temperature)

① 可变写法: [1, 2, 3, 4]
② 不可变写法: 原 = [1, 2, 3] | 新 = [1, 2, 3, 4]
③ 不可变字典合并: 原 = {'a': 1} | 新 = {'a': 1, 'b': 2}
④ frozen 实例: ModelConfig(name='gpt', temperature=0.7)
⑤ 触发报错: FrozenInstanceError - cannot assign to field 'temperature'
⑥ replace 生成新实例: 原 = 0.7 | 新 = 1.5


---
# 四、高阶函数：map / filter / reduce

**本章解决什么问题**：把「遍历 + 判断 + 累积」这三类循环抽象成三个通用工具。

**三大高阶函数对比**

| 函数 | 作用 | 输入 → 输出 | 返回类型 | 更地道的替代品 |
|---|---|---|---|---|
| `map(f, it)` | 逐元素变换 | `iterable` → `iterator` | 惰性 | 推导式 `[f(x) for x in it]` |
| `filter(f, it)` | 按条件保留 | `iterable` → `iterator` | 惰性 | `[x for x in it if f(x)]` |
| `reduce(f, it)` | 两两归约成单值 | `iterable` → 单个值 | 立即 | `sum / max / any / all` 等 |

> `map`/`filter` 接收函数为参数，`reduce` 也是 —— 三者都是高阶函数；
> 自定义高阶函数（如 `compose`）见 4.4 节。

In [7]:
# ---------- 第 4 章：map / filter / reduce ----------
# 4.1 map：对每个元素应用函数（惰性，需 list() 物化查看）
nums = [1, 2, 3, 4]
squares = list(map(lambda x: x ** 2, nums))
print("① map 求平方:", squares)          # [1, 4, 9, 16]

# 4.2 map 接收多个可迭代对象：按位置并行取参（最短者决定长度）
a = [1, 2, 3]
b = [10, 20, 30]
print("② map 并行相加:", list(map(lambda x, y: x + y, a, b)))   # [11, 22, 33]

# 4.3 filter：保留使函数为真的元素
nums = [1, 2, 3, 4, 5, 6]
print("③ filter 取偶数:", list(filter(lambda x: x % 2 == 0, nums)))   # [2, 4, 6]

# filter(None, it)：None 表示「保留真值」，常用于清洗空数据
data = [0, 1, "", "hello", None, [], [1]]
print("④ filter(None, ...) 去假值:", list(filter(None, data)))   # [1, 'hello', [1]]

# 4.4 reduce：累积归约（注意从 functools 导入，Python3 不再是内置）
from functools import reduce
nums = [1, 2, 3, 4, 5]
print("⑤ reduce 求和:", reduce(lambda acc, x: acc + x, nums))            # 15
print("⑥ reduce 带初始值:", reduce(lambda acc, x: acc + x, nums, 100))    # 115
print("⑦ reduce 求最大:", reduce(lambda p, q: p if p > q else q, nums))  # 5
print("⑧ 但求最大值更应使用内置 max:", max(nums))

# 4.5 自定义高阶函数：接收函数、返回函数
def compose(f, g):                 # compose 本身也是高阶函数
    return lambda x: f(g(x))

def double(x):
    return x * 2

def increment(x):
    return x + 1

h = compose(double, increment)      # h(x) = double(increment(x))
print("⑨ compose(double, increment)(3) =", h(3))    # 8

① map 求平方: [1, 4, 9, 16]
② map 并行相加: [11, 22, 33]
③ filter 取偶数: [2, 4, 6]
④ filter(None, ...) 去假值: [1, 'hello', [1]]
⑤ reduce 求和: 15
⑥ reduce 带初始值: 115
⑦ reduce 求最大: 5
⑧ 但求最大值更应使用内置 max: 5
⑨ compose(double, increment)(3) = 8


---
# 五、lambda 表达式

**本章解决什么问题**：什么时候用匿名函数最划算，什么时候它是负担。

**语法**：`lambda 参数: 表达式`（只能是**一个表达式**，其值即返回值）

**限制（硬性）**

- 只能写表达式，不能写语句；
- 不能包含 `return` / `yield` / `assert`（它们都是语句）；
- 复杂逻辑（条件超过一层、需要循环）应改用 `def`。

**典型应用**：`sorted` / `sort` 的 `key` 参数 —— 传一个「小函数」最方便的地方。

In [8]:
# ---------- 第 5 章：lambda 的用法与边界 ----------
add = lambda a, b: a + b
print("① 最简 lambda:", add(1, 2))    # 3

# 5.1 排序：key 接收一个函数，lambda 是最轻量的写法
words = ["banana", "apple", "cherry"]
words.sort(key=lambda w: len(w))
print("② 按长度排序:", words)         # ['apple', 'banana', 'cherry']

# 5.2 多级排序：key 返回元组，先比第一项、再比第二项
users = [
    {"name": "Alice", "age": 30},
    {"name": "Bob",   "age": 25},
    {"name": "Carol", "age": 30},
]
users.sort(key=lambda u: (u["age"], u["name"]))
print("③ 先按年龄再按姓名:", [u["name"] for u in users])   # ['Bob', 'Alice', 'Carol']

# 5.3 验证「lambda 不能写语句」：用 compile 触发 SyntaxError（反例）
try:
    compile("lambda x: return x", "<demo>", "eval")   # 错误写法：return 是语句
except SyntaxError as e:
    print("④ lambda 内写 return 触发:", type(e).__name__)

# 5.4 风格边界：单条件表达式可以，但建议只用于「一眼能看懂」的场景
f = lambda x: x if x > 0 else -x     # 不推荐：PEP 8 建议始终绑定名字用 def
def abs_value(x):                    # 推荐写法
    return x if x > 0 else -x
print("⑤ 两者行为一致:", f(-7), abs_value(-7), "但 def 版本可读、可测试、可加文档")

① 最简 lambda: 3
② 按长度排序: ['apple', 'banana', 'cherry']
③ 先按年龄再按姓名: ['Bob', 'Alice', 'Carol']
④ lambda 内写 return 触发: SyntaxError
⑤ 两者行为一致: 7 7 但 def 版本可读、可测试、可加文档


---
# 六、闭包与工厂函数

**本章解决什么问题**：函数怎么「记住」自己定义时的环境？—— 这是装饰器、配置化函数的原理。

**闭包 = 内层函数 + 它引用的外层局部变量**

- 工厂函数：返回配置好的函数（`multiplier(2)` 产出 `double`）；
- 计数器：闭包封装私有状态（比全局变量安全）；
- 配置化函数：一次生成多个「预配置」的变体。

| 概念 | 关键语法 | 典型代码 |
|---|---|---|
| 工厂函数 | 外层返回内层函数 | `multiplier(2)` |
| 读写闭包变量 | 内层用 `nonlocal` | `make_counter()` |
| 查看闭包变量 | `fn.__closure__` | 调试利器 |

In [9]:
# ---------- 第 6 章：闭包 ----------
# 6.1 工厂函数：内层函数记住了外层参数 n
def multiplier(n):
    def mul(x):
        return x * n      # n 来自外层作用域（闭包捕获）
    return mul

double = multiplier(2)
triple = multiplier(3)
print("① double(5) =", double(5))    # 10
print("② triple(5) =", triple(5))   # 15
print("③ 闭包里捕获的 n:", double.__closure__[0].cell_contents)

# 6.2 计数器：nonlocal 让内层函数可以「写」外层变量
def make_counter():
    count = 0
    def counter():
        nonlocal count     # 声明写权限，否则会报 UnboundLocalError（反例见后文）
        count += 1
        return count
    return counter

c = make_counter()
print("④ 连续计数:", c(), c())        # 1 2

# 6.3 配置化函数：LLMOps 日志器按前缀生成变体
def with_prefix(prefix):
    def log(msg):
        return f"[{prefix}] {msg}"     # 这里改为 return（保证纯函数风格，也便于 print 验证）
    return log

info = with_prefix("INFO")
error = with_prefix("ERROR")
print("⑤", info("启动"))              # [INFO] 启动
print("⑥", error("失败"))             # [ERROR] 失败

# 6.4 两个计数器互不干扰 —— 每次调用工厂都创建独立闭包
c1, c2 = make_counter(), make_counter()
c1(); c1()
print("⑦ c1 =", c1(), " c2 =", c2(), "（独立状态）")

① double(5) = 10
② triple(5) = 15
③ 闭包里捕获的 n: 2
④ 连续计数: 1 2
⑤ [INFO] 启动
⑥ [ERROR] 失败
⑦ c1 = 3  c2 = 1 （独立状态）


---
# 七、functools 工具箱

**本章解决什么问题**：标准库已经把偏应用、缓存、分派、装饰器元信息这些高频需求做完了，直接用。

**七大工具速查**

| 工具 | 一句话用途 | 关键点 |
|---|---|---|
| `partial(f, *a, **k)` | 固定部分参数生成新函数 | 这是**偏函数应用**（不是柯里化，见第十章修正说明） |
| `lru_cache(maxsize)` | 按参数缓存返回值（LRU 淘汰） | 参数必须**可哈希** |
| `cache` | `lru_cache(maxsize=None)` 的 3.9+ 简写 | 无限缓存 |
| `reduce` | 归约（见第 4 章） | — |
| `singledispatch` | 按第一个参数类型分派实现 | 注册多个同名函数体 |
| `wraps` | 装饰器中保留原函数元信息 | 写装饰器必加 |
| `total_ordering` | 只写 `__eq__`+`__lt__`，补全其余比较 | 类装饰器 |

In [10]:
# ---------- 第 7 章：functools（上）：partial / lru_cache / cache ----------
from functools import partial, lru_cache, cache

# 7.1 partial：固定部分参数。LLMOps 场景：从通用 API 客户端派生专用客户端
def power(base, exp):
    return base ** exp

square = partial(power, exp=2)    # 关键字固定
cube = partial(power, exp=3)
print("① square(5) =", square(5))   # 25
print("② cube(5)   =", cube(5))     # 125
print("③ partial 产物的类型:", type(square).__name__, "| 保留了原函数:",
      square.func.__name__, square.keywords)

# 7.2 lru_cache：斐波那契 —— 从指数级降为线性
@lru_cache(maxsize=None)
def fib(n):
    if n < 2:
        return n
    return fib(n - 1) + fib(n - 2)

print("④ fib(100) =", fib(100))      # 354224848179261915075
print("⑤ 缓存命中情况:", fib.cache_info())

# 7.3 cache：Python 3.9+ 等价于 @lru_cache(maxsize=None)
@cache
def fib2(n):
    return n if n < 2 else fib2(n - 1) + fib2(n - 2)
print("⑥ @cache 版 fib2(100) =", fib2(100))

# 7.4 验证「可缓存」的前提是纯函数：同参数第二次调用直接命中缓存、不再执行函数体
@lru_cache(maxsize=None)
def loud(n):
    print(f"   >>> 真正计算了 loud({n})")
    return n * 10

print("⑦ 第一次调用:"); print("   返回 =", loud(3))
print("⑧ 第二次调用:"); print("   返回 =", loud(3), "（函数体没有执行）")

① square(5) = 25
② cube(5)   = 125
③ partial 产物的类型: partial | 保留了原函数: power {'exp': 2}
④ fib(100) = 354224848179261915075
⑤ 缓存命中情况: CacheInfo(hits=98, misses=101, maxsize=None, currsize=101)
⑥ @cache 版 fib2(100) = 354224848179261915075
⑦ 第一次调用:
   >>> 真正计算了 loud(3)
   返回 = 30
⑧ 第二次调用:
   返回 = 30 （函数体没有执行）


In [11]:
# ---------- 第 7 章：functools（下）：singledispatch / wraps / total_ordering ----------
from functools import singledispatch, wraps, total_ordering

# 7.5 singledispatch：按第一个参数类型选择实现（替代 if isinstance 链）
@singledispatch
def process(x):
    raise TypeError(f"不支持 {type(x)}")

@process.register(int)
def _(x):
    return x * 2

@process.register(str)
def _(x):
    return x.upper()

print("① process(10)   =", process(10))    # 20
print("② process('hi')  =", process("hi")) # HI

# 7.6 wraps：装饰器保留原函数元信息（不加会丢失 __name__ 和文档）
def log(func):
    @wraps(func)                # 关键：把 func 的元信息拷贝给 wrapper
    def wrapper(*args, **kwargs):
        return f"调用 {func.__name__} -> {func(*args, **kwargs)}"
    return wrapper

@log
def get_answer():
    """返回终极答案。"""
    return 42

print("③", get_answer())
print("④ __name__ 被保留 =", get_answer.__name__, "| doc =", get_answer.__doc__)

# 7.7 total_ordering：只实现 __eq__ + __lt__，自动补全 <= > >=
@total_ordering
class Version:
    def __init__(self, major, minor):
        self.major, self.minor = major, minor
    def __eq__(self, other):
        return (self.major, self.minor) == (other.major, other.minor)
    def __lt__(self, other):
        return (self.major, self.minor) < (other.major, other.minor)
    def __repr__(self):
        return f"{self.major}.{self.minor}"

print("⑤ 排序:", sorted([Version(1, 9), Version(1, 2), Version(0, 9)]))
print("⑥ 自动补全的比较: 1.2 <= 1.9 ->", Version(1, 2) <= Version(1, 9))

① process(10)   = 20
② process('hi')  = HI
③ 调用 get_answer -> 42
④ __name__ 被保留 = get_answer | doc = 返回终极答案。
⑤ 排序: [0.9, 1.2, 1.9]
⑥ 自动补全的比较: 1.2 <= 1.9 -> True


---
# 八、itertools 工具箱

**本章解决什么问题**：迭代器层面的「乐高积木」——拼接、组合、切片、分组、累积，全部惰性执行。

**五组工具速查**

| 类别 | 工具 | 作用 |
|---|---|---|
| 无限迭代器 | `count / cycle / repeat` | 产生无限序列（必须配合 `islice`/`break` 使用） |
| 组合迭代器 | `chain / zip_longest / product / permutations / combinations` | 拼接与笛卡尔积/排列/组合 |
| 过滤切片 | `islice / takewhile / dropwhile / filterfalse` | 惰性切片与条件截断 |
| 分组 | `groupby` | 按 key 分组（**必须先排序**） |
| 累积 | `accumulate` | 前缀累积（和、积、任意二元函数） |

In [12]:
# ---------- 第 8 章（上）：无限迭代器 + 组合迭代器 ----------
from itertools import count, cycle, repeat, chain, zip_longest, product, permutations, combinations

# 8.1 无限迭代器：务必配 break / islice，否则永不停止
for i in count(10):                  # count(10) 从 10 开始无限 +1
    if i > 13:
        break
    print("① count:", i)            # 10 11 12 13

for i, c in enumerate(cycle("AB")):  # cycle 无限重复 "AB"
    if i >= 4:
        break
    print("② cycle:", c)            # A B A B

print("③ repeat:", list(repeat("x", 3)))   # ['x', 'x', 'x']

# 8.2 组合迭代器
print("④ chain 拼接:", list(chain([1, 2], [3, 4])))                      # [1, 2, 3, 4]
print("⑤ zip_longest 补齐:", list(zip_longest([1, 2], ["a"], fillvalue="-")))
# [(1, 'a'), (2, '-')]
print("⑥ product 笛卡尔积:", list(product([1, 2], "ab")))
# [(1, 'a'), (1, 'b'), (2, 'a'), (2, 'b')]
print("⑦ permutations 排列:", list(permutations([1, 2, 3], 2)))
print("⑧ combinations 组合:", list(combinations([1, 2, 3], 2)))

① count: 10
① count: 11
① count: 12
① count: 13
② cycle: A
② cycle: B
② cycle: A
② cycle: B
③ repeat: ['x', 'x', 'x']
④ chain 拼接: [1, 2, 3, 4]
⑤ zip_longest 补齐: [(1, 'a'), (2, '-')]
⑥ product 笛卡尔积: [(1, 'a'), (1, 'b'), (2, 'a'), (2, 'b')]
⑦ permutations 排列: [(1, 2), (1, 3), (2, 1), (2, 3), (3, 1), (3, 2)]
⑧ combinations 组合: [(1, 2), (1, 3), (2, 3)]


In [13]:
# ---------- 第 8 章（下）：过滤切片 / 分组 / 累积 ----------
from itertools import islice, takewhile, dropwhile, filterfalse, groupby, accumulate

# 8.3 过滤与切片（全部惰性，可以安全地配合无限迭代器）
print("① islice 截取无限流:", list(islice(count(), 5)))          # [0, 1, 2, 3, 4]
print("② takewhile 条件截断:", list(takewhile(lambda x: x < 5, count())))  # [0..4]
print("③ dropwhile 条件后取:", list(dropwhile(lambda x: x < 5, range(10)))) # [5..9]
print("④ filterfalse 反向过滤:", list(filterfalse(lambda x: x % 2, range(10))))  # 偶数

# 8.4 groupby：相邻同 key 分组 —— 数据必须先按 key 排序（经典陷阱，见陷阱章节）
data = [
    {"type": "a", "value": 1},
    {"type": "a", "value": 2},
    {"type": "b", "value": 3},
]
data.sort(key=lambda x: x["type"])            # 不排序会得到错乱的分组
for key, group in groupby(data, key=lambda x: x["type"]):
    print("⑤ groupby:", key, [item["value"] for item in group])

# 8.5 accumulate：前缀累积
print("⑥ 累加:", list(accumulate([1, 2, 3, 4])))                       # [1, 3, 6, 10]
print("⑦ 累乘:", list(accumulate([1, 2, 3, 4], lambda a, b: a * b))) # [1, 2, 6, 24]

# 8.6 验证惰性：无限流 + islice 组合不会卡死
print("⑧ 无限流安全截取:", list(islice(cycle([1, 2, 3]), 7)))

① islice 截取无限流: [0, 1, 2, 3, 4]
② takewhile 条件截断: [0, 1, 2, 3, 4]
③ dropwhile 条件后取: [5, 6, 7, 8, 9]
④ filterfalse 反向过滤: [0, 2, 4, 6, 8]
⑤ groupby: a [1, 2]
⑤ groupby: b [3]
⑥ 累加: [1, 3, 6, 10]
⑦ 累乘: [1, 2, 6, 24]
⑧ 无限流安全截取: [1, 2, 3, 1, 2, 3, 1]


---
# 九、函数组合：compose 与 pipe

**本章解决什么问题**：把 `f(g(h(x)))` 这种嵌套调用，改造成**从右到左（compose）**或**从左到右（pipe）**的线性流水线。

**两种方向对比**

| 风格 | 读法 | 等价数学式 | 适用 |
|---|---|---|---|
| `compose(f, g)(x)` | 先 g 后 f | `f∘g(x)` | 数学传统、装饰器叠加 |
| `pipe(x, f, g)` | 先 f 后 g | `g(f(x))` | 数据处理管道（更贴近阅读顺序） |

> **执行顺序验证**：`compose(double, increment)(3)` = `double(increment(3))` = `(3+1)*2` = `8`，
> 即 compose 的**右侧参数先执行**。

In [14]:
# ---------- 第 9 章：compose / pipe / reduce 实现 ----------
# 9.1 手动 compose：*funcs 接收任意多个函数
def compose(*funcs):
    def composed(x):
        for f in reversed(funcs):   # 关键：reversed -> 最右边的函数最先执行
            x = f(x)
        return x
    return composed

def double(x):
    return x * 2

def increment(x):
    return x + 1

f = compose(double, increment)
print("① compose(double, increment)(3) =", f(3))   # 8：先 +1 再 *2

# 9.2 pipe：数据在前、函数依次排列（读起来像 Unix 管道）
def pipe(value, *funcs):
    for f in funcs:
        value = f(value)
    return value

result = pipe(
    5,
    lambda x: x + 1,   # 6
    lambda x: x * 2,   # 12
    str,               # "12"
)
print("② pipe 结果:", repr(result))   # '12'

# 9.3 用 reduce 实现一行版 compose（函数式到极致的写法）
from functools import reduce

def compose_r(*funcs):
    return lambda x: reduce(lambda acc, f: f(acc), reversed(funcs), x)

g = compose_r(double, increment, increment)
print("③ reduce 版 compose(3) =", g(3))   # (3+1+1)*2 = 10

# 9.4 验证执行顺序的论断：在函数里插入探针
def traced(name, fn):
    def wrapper(x):
        print(f"   [{name}] 收到 {x} -> 返回 {fn(x)}")
        return fn(x)
    return wrapper

f2 = compose(traced("double", double), traced("increment", increment))
print("④ 调用 f2(3)，观察执行顺序：")
f2(3)   # 输出证明 increment 先执行

① compose(double, increment)(3) = 8
② pipe 结果: '12'
③ reduce 版 compose(3) = 10
④ 调用 f2(3)，观察执行顺序：
   [increment] 收到 3 -> 返回 4
   [double] 收到 4 -> 返回 8


8

---
# 十、柯里化与偏函数应用

**本章解决什么问题**：把多参数函数拆成「可分步喂参数」的形态，实现参数复用与延迟执行。

> ### ⚠️ 修正说明（源文档技术性偏差）
> 源文档第 10 章把 `functools.partial` 的示例称为「通用柯里化」，这是**不准确的**：
>
> | 概念 | 定义 | `partial(add, 1, 2)` | `curry(add)(1)(2)(3)` |
> |---|---|---|---|
> | **偏函数应用** | 一次性固定若干参数，剩余参数一次传完 | `add_ab(3)` 一次传 1 个参数 | — |
> | **柯里化** | 把 n 参函数变换为「每次只接收 1 个参数」的嵌套函数链 | — | `(1)(2)(3)` 逐个传 |
>
> `partial` 是**偏函数应用**；真正的柯里化用嵌套 lambda / 闭包实现。下面用代码验证两者行为差异。

In [15]:
# ---------- 第 10 章：柯里化 vs 偏函数应用（含修正说明验证） ----------
# 10.1 手写柯里化：多参数 -> 一系列单参数函数
def curry_add(a):
    return lambda b: a + b        # 每一步只接收 1 个参数

add5 = curry_add(5)
print("① curry_add(5)(3) =", add5(3))    # 8

# 10.2 偏函数应用（partial）：一次固定多个参数，剩下的**一次传完**
from functools import partial

def add(a, b, c):
    return a + b + c

add_ab = partial(add, 1, 2)
print("② partial(add, 1, 2)(3) =", add_ab(3))   # 6

# 10.3 【修正说明验证】两者对 add(1,2,3) 的调用形态不同：
#    柯里化要求逐个传参 —— 一次性传 3 个会报错（错误写法演示）
def curry3(f):
    def a(x):
        def b(y):
            def c(z):
                return f(x, y, z)
            return c
        return b
    return a

curried_add = curry3(add)
try:
    curried_add(1, 2, 3)          # 错误写法：柯里化后不能一次传多个参数
except TypeError as e:
    print("③ 柯里化后一次性传参触发:", type(e).__name__, "-", e)
print("④ 正确的柯里化调用: curried_add(1)(2)(3) =", curried_add(1)(2)(3))
print("⑤ partial 版调用:        add_ab(3)      =", add_ab(3))
print("⑥ 结论：partial=偏应用（剩余参数一次传），柯里化=严格逐参嵌套，二者不等价")

① curry_add(5)(3) = 8
② partial(add, 1, 2)(3) = 6
③ 柯里化后一次性传参触发: TypeError - curry3.<locals>.a() takes 1 positional argument but 3 were given
④ 正确的柯里化调用: curried_add(1)(2)(3) = 6
⑤ partial 版调用:        add_ab(3)      = 6
⑥ 结论：partial=偏应用（剩余参数一次传），柯里化=严格逐参嵌套，二者不等价


---
# 十一、递归

**本章解决什么问题**：函数式语言用递归代替循环（不可变计数器），但 Python 对递归支持有限，要清楚边界。

**要点**

- 阶乘 / 斐波那契 / 树遍历是三大经典递归；
- 递归 + `lru_cache` = 记忆化，避免重复子问题计算；
- **Python 不支持尾递归优化**：即使写成尾递归形式，依然消耗栈；
- 默认递归深度约 1000（`sys.getrecursionlimit()`），深递归应改用循环或生成器。

In [16]:
# ---------- 第 11 章：递归 ----------
import sys
from functools import lru_cache

# 11.1 阶乘：递归定义最直观
def factorial(n):
    return 1 if n <= 1 else n * factorial(n - 1)
print("① factorial(5) =", factorial(5))    # 120

# 11.2 斐波那契 + 记忆化（无缓存是指数级，有缓存是线性）
@lru_cache(maxsize=None)
def fib(n):
    return n if n < 2 else fib(n - 1) + fib(n - 2)
print("② fib(50) =", fib(50))             # 12586269025

# 11.3 尾递归形式：Python 不做尾调用优化，仍然消耗栈 —— 用栈溢出验证
def factorial_tail(n, acc=1):
    return acc if n <= 1 else factorial_tail(n - 1, acc * n)
print("③ 小规模尾递归可用:", factorial_tail(5))
try:
    factorial_tail(10000)          # 错误写法：深递归（尾递归形式也一样会爆栈）
except RecursionError as e:
    print("④ 深度 10000 触发:", type(e).__name__, "-> Python 无尾递归优化")
print("⑤ 当前递归限制:", sys.getrecursionlimit())

# 11.4 树遍历：递归的舒适区（这里用简单的嵌套 dict 模拟树）
tree = {"value": 1,
        "left":  {"value": 2, "left": None, "right": None},
        "right": {"value": 3, "left": None, "right": None}}

def tree_sum(node):
    if node is None:
        return 0
    return node["value"] + tree_sum(node["left"]) + tree_sum(node["right"])

print("⑥ tree_sum =", tree_sum(tree))     # 6

① factorial(5) = 120
② fib(50) = 12586269025
③ 小规模尾递归可用: 120
④ 深度 10000 触发: RecursionError -> Python 无尾递归优化
⑤ 当前递归限制: 1000
⑥ tree_sum = 6


---
# 十二、生成器与惰性求值

**本章解决什么问题**：处理大数据 / 无限流时，如何做到「一次只占一份内存」？—— 生成器管道。

**要点**

- `yield` 让函数变成生成器：**调用时不执行、取值时才执行**（前置知识已验证）；
- 生成器管道 = 多个生成器串联，数据像水流一样逐条通过；
- 无限序列（自然数、斐波那契）只有配 `islice` 才能安全使用；
- LLMOps 场景：读取大文件 / 流式处理 LLM 响应时的标准姿势。

In [17]:
# ---------- 第 12 章：生成器与惰性求值 ----------
from itertools import islice
from pathlib import Path

# 12.1 无限自然数序列
def naturals():
    n = 0
    while True:
        yield n
        n += 1

print("① naturals 前 5 个:", list(islice(naturals(), 5)))   # [0, 1, 2, 3, 4]

# 12.2 生成器管道：读文件 -> 过滤空行 -> 转大写（惰性，逐行流过）
def read_lines(path):
    with open(path, encoding="utf-8") as f:
        for line in f:
            yield line.rstrip()

def filter_empty(lines):
    for line in lines:
        if line:
            yield line

def to_upper(lines):
    for line in lines:
        yield line.upper()

# 准备演示文件（真实读取，用完即删，保证单元可重复执行）
demo_file = Path("data_demo.txt")
demo_file.write_text("  hello  \n\n  world  \n", encoding="utf-8")

pipeline = to_upper(filter_empty(read_lines(demo_file)))
print("② 管道产物类型:", type(pipeline).__name__, "（还没开始读文件）")
for line in pipeline:
    print("③", repr(line))           # 'HELLO'  'WORLD'
demo_file.unlink()                   # 清理临时文件

# 12.3 无限斐波那契流
def fib_gen():
    a, b = 0, 1
    while True:
        yield a
        a, b = b, a + b

print("④ 斐波那契前 10 项:", list(islice(fib_gen(), 10)))

① naturals 前 5 个: [0, 1, 2, 3, 4]
② 管道产物类型: generator （还没开始读文件）
③ '  HELLO'
③ '  WORLD'
④ 斐波那契前 10 项: [0, 1, 1, 2, 3, 5, 8, 13, 21, 34]


---
# 十三、函数式 vs 命令式（三种写法怎么选）

**本章解决什么问题**：同一个任务有三种写法，到底哪种是「Python 之禅」？

| 写法 | 思路 | 可读性 | 性能 | 建议 |
|---|---|---|---|---|
| 命令式 for 循环 | 空列表 + append | 步骤清晰但啰嗦 | 好 | 逻辑复杂、有多个中间状态时用 |
| map/filter 函数式 | 高阶函数组合 | 一般 | 与推导式相当 | 已有现成函数（如 `str.upper`）时用 |
| **推导式** | 声明式一行表达 | **最好** | **最好** | **默认选择（Python 风格）** |

> 经验法则：**简单变换用推导式；需要提前退出/复杂分支用 for；map 只有在直接传函数名时比 lambda 推导式更优雅。**

In [18]:
# ---------- 第 13 章（源 14）：三种写法对照 ----------
# 1) 命令式
result_cmd = []
for x in range(10):
    if x % 2 == 0:
        result_cmd.append(x ** 2)

# 2) 函数式（map + filter）
result_fp = list(map(lambda x: x ** 2, filter(lambda x: x % 2 == 0, range(10))))

# 3) 推导式（Python 风格）
result_comp = [x ** 2 for x in range(10) if x % 2 == 0]

print("① 命令式:", result_cmd)
print("② 函数式:", result_fp)
print("③ 推导式:", result_comp)

# 等价性验证：三种写法结果完全一致（对「等价性论断」的代码证明）
print("④ 三者相等:", result_cmd == result_fp == result_comp)

# map 直接传现成函数时最优雅（无需 lambda）
words = ["hello", "world"]
print("⑤ map(str.upper):", list(map(str.upper, words)))

① 命令式: [0, 4, 16, 36, 64]
② 函数式: [0, 4, 16, 36, 64]
③ 推导式: [0, 4, 16, 36, 64]
④ 三者相等: True
⑤ map(str.upper): ['HELLO', 'WORLD']


---
# 十四、反例单元：亲手触发每一种典型错误

**本章解决什么问题**：与其背「不要这样写」，不如亲手触发一次报错、看清症状，再对照正确写法。

> 以下每个反例都**真实执行**了错误写法并用 `try/except` 捕获报错（保证整个 Notebook 0 报错输出），
> 随后给出正确写法并对比输出差异。

In [19]:
# ---------- 反例 1：lambda 延迟绑定（循环变量捕获） ----------
# ❌ 错误写法：所有 lambda 共享同一个循环变量 i
funcs_bad = [lambda: i for i in range(3)]
print("❌ 错误输出:", [f() for f in funcs_bad])   # [2, 2, 2]  全是最后的 i

# ✅ 正确写法：用默认参数在定义时立即绑定当前值
funcs_good = [lambda i=i: i for i in range(3)]
print("✅ 正确输出:", [f() for f in funcs_good])   # [0, 1, 2]

# 原理验证：错误版本所有 lambda 捕获的是「同一个」闭包变量（迭代结束后停留在 2）
print("原理: 三个错误 lambda 闭包里的同一个变量 =",
      funcs_bad[0].__closure__[0].cell_contents,
      "（推导式作用域的 i 迭代完停在 2，三个 lambda 都引用它）")

❌ 错误输出: [2, 2, 2]
✅ 正确输出: [0, 1, 2]
原理: 三个错误 lambda 闭包里的同一个变量 = 2 （推导式作用域的 i 迭代完停在 2，三个 lambda 都引用它）


In [20]:
# ---------- 反例 2：可变默认参数 ----------
# ❌ 错误写法：默认值 [] 在函数定义时只创建一次，之后跨调用共享
def append_bad(item, lst=[]):
    lst.append(item)
    return lst

print("❌ 第一次调用:", append_bad(1))   # [1]  看似正常
print("❌ 第二次调用:", append_bad(2))   # [1, 2]  上次的 1 还在 —— 状态泄漏！

# ✅ 正确写法：默认 None，函数体内新建
def append_good(item, lst=None):
    if lst is None:
        lst = []
    lst.append(item)
    return lst

print("✅ 第一次调用:", append_good(1))
print("✅ 第二次调用:", append_good(2))   # [2]  每次都是新列表

# 验证「定义时只创建一次」：两次调用后返回的长度递增 -> 默认列表被跨调用共享
def probe(x, memo=[]):
    memo.append(x)
    return len(memo)
print("原理: probe 两次调用返回的长度 =", probe(1), "和", probe(2),
      "（长度递增 => 两次 append 落在同一个默认列表上）")

❌ 第一次调用: [1]
❌ 第二次调用: [1, 2]
✅ 第一次调用: [1]
✅ 第二次调用: [2]
原理: probe 两次调用返回的长度 = 1 和 2 （长度递增 => 两次 append 落在同一个默认列表上）


In [21]:
# ---------- 反例 3：map/filter 返回的是一次性迭代器 ----------
# ❌ 错误写法：把 map 结果当列表反复使用
result = map(lambda x: x * 2, [1, 2, 3])
print("❌ 直接 print:", result)          # <map object ...> 本体不可读
print("❌ 第一次物化:", list(result))    # [2, 4, 6]
print("❌ 第二次物化:", list(result))    # []  已耗尽！迭代器只能消费一次

# ✅ 正确写法：需要多次使用就先物化成列表
result_ok = list(map(lambda x: x * 2, [1, 2, 3]))
print("✅ 物化后可反复使用:", result_ok, result_ok)

# filter 同理的验证
fres = filter(None, [0, 1, "", "a"])
print("✅ filter 第一次:", list(fres), "| 第二次:", list(fres))

❌ 直接 print: <map object at 0x107d9b2b0>
❌ 第一次物化: [2, 4, 6]
❌ 第二次物化: []
✅ 物化后可反复使用: [2, 4, 6] [2, 4, 6]
✅ filter 第一次: [1, 'a'] | 第二次: []


In [22]:
# ---------- 反例 4：lru_cache 参数必须可哈希 ----------
from functools import lru_cache

# ❌ 错误写法：可变参数（list/dict）无法作为缓存 key
@lru_cache(maxsize=None)
def process_list(lst):
    return sum(lst)

try:
    process_list([1, 2, 3])        # 错误写法：list 不可哈希
except TypeError as e:
    print("❌ 触发报错:", type(e).__name__, "-", e)

# ✅ 正确写法：改用可哈希的 tuple
@lru_cache(maxsize=None)
def process_tuple(t):
    return sum(t)

print("✅ tuple 版正常且被缓存:", process_tuple((1, 2, 3)), process_tuple.cache_info())

❌ 触发报错: TypeError - unhashable type: 'list'
✅ tuple 版正常且被缓存: 6 CacheInfo(hits=0, misses=1, maxsize=None, currsize=1)


In [23]:
# ---------- 反例 5：闭包内不加 nonlocal 直接赋值 ----------
# ❌ 错误写法：内层函数对同名变量赋值 -> 被视为局部变量
def make_bad_counter():
    count = 0
    def inc():
        count += 1        # UnboundLocalError：count 被当成 inc 的局部变量
        return count
    return inc

bad = make_bad_counter()
try:
    bad()                  # 错误写法：实际调用触发报错
except UnboundLocalError as e:
    print("❌ 触发报错:", type(e).__name__, "-", e)

# ✅ 正确写法：nonlocal 声明写权限
def make_good_counter():
    count = 0
    def inc():
        nonlocal count     # 关键一行
        count += 1
        return count
    return inc

good = make_good_counter()
print("✅ 正确输出:", good(), good(), good())

❌ 触发报错: UnboundLocalError - cannot access local variable 'count' where it is not associated with a value
✅ 正确输出: 1 2 3


In [24]:
# ---------- 反例 6：groupby 不排序 + 修改 frozen 实例 + 深递归（三连演示） ----------
from itertools import groupby
from dataclasses import dataclass, replace

# 6a) ❌ groupby 未排序：相邻才分组，同 key 出现两段
data = [{"t": "a"}, {"t": "b"}, {"t": "a"}]      # 故意不排序
groups_bad = [(k, len(list(g))) for k, g in groupby(data, key=lambda x: x["t"])]
print("❌ 未排序分组:", groups_bad)               # [('a',1),('b',1),('a',1)] 同 key 被拆散

data.sort(key=lambda x: x["t"])                  # ✅ 先排序
groups_good = [(k, len(list(g))) for k, g in groupby(data, key=lambda x: x["t"])]
print("✅ 排序后分组:", groups_good)             # [('a',2),('b',1)]

# 6b) ❌ 修改 frozen dataclass
@dataclass(frozen=True)
class Point:
    x: float
    y: float

p = Point(1, 2)
try:
    p.x = 3                    # 错误写法
except Exception as e:
    print("❌ FrozenInstanceError:", e)
p2 = replace(p, x=3)           # ✅ 正确写法：返回新实例
print("✅ replace 新实例:", p2, "| 原实例不变:", p)

# 6c) ❌ 深递归爆栈（RecursionError 用 try 捕获验证）
def deep(n):
    return 0 if n == 0 else deep(n - 1) + 1
try:
    deep(5000)                 # 错误写法：超过默认递归深度
except RecursionError as e:
    print("❌ RecursionError:", e)

def deep_iter(n):              # ✅ 正确写法：循环代替深递归
    total = 0
    for i in range(n):
        total += 1
    return total
print("✅ 循环版 5000 层深度:", deep_iter(5000))

❌ 未排序分组: [('a', 1), ('b', 1), ('a', 1)]
✅ 排序后分组: [('a', 2), ('b', 1)]
❌ FrozenInstanceError: cannot assign to field 'x'
✅ replace 新实例: Point(x=3, y=2) | 原实例不变: Point(x=1, y=2)
❌ RecursionError: maximum recursion depth exceeded
✅ 循环版 5000 层深度: 5000


---
# 十五、实用示例集：5 个工程高频模式（LLMOps 视角）

**本章解决什么问题**：函数式编程在业务代码里到底长什么样？—— 5 个可以直接抄走的模式。

| # | 模式 | 用到的知识点 | 适用场景 | 注意点 |
|---|---|---|---|---|
| 1 | **数据清洗管道** | compose/pipe + 纯函数 | Prompt / 日志文本清洗 | 每步保持纯函数，方便单测 |
| 2 | **重试装饰器组合** | 闭包 + wraps + 装饰器叠加 | 调用不稳定的外部 LLM API | 重试参数用工厂函数外置 |
| 3 | **记忆化缓存** | lru_cache + 纯函数 | 相同请求直接返回缓存结果 | 参数必须可哈希；缓存要设上限 |
| 4 | **流式管道（生成器）** | 生成器 + islice | 大文件 / 流式响应处理 | 迭代器只能消费一次 |
| 5 | **不可变更新 + 状态机** | dict 展开 + 函数即状态 | 请求生命周期管理、配置热更新 | 状态迁移用纯函数表达 |

In [25]:
# ---------- 模式 1：数据清洗管道（Prompt 预处理） ----------
def pipe(value, *funcs):
    for f in funcs:
        value = f(value)
    return value

# 每一步都是纯函数：可以单独测试、任意组合
def strip(s):        return s.strip()
def lower(s):        return s.lower()
def collapse(s):     return " ".join(s.split())      # 压缩多余空白

def clean(text):
    return pipe(text, strip, lower, collapse)

# > 【修正说明】源文档 13.1 的写法 `[clean(s) for s in data if clean(s)]`
# > 会让每个元素执行两遍 clean（副作用翻倍、性能浪费）。
# > 修正为「先 map 再 filter」，并验证 clean 只执行一遍。
call_count = {"n": 0}
def clean_counted(s):
    call_count["n"] += 1
    return clean(s)

data = ["  Hello   World  ", "", "  LLM Ops  "]
cleaned = list(filter(None, map(clean_counted, data)))
print("① 清洗结果:", cleaned)
print("② clean 执行次数:", call_count["n"], "次（元素数", len(data), "个，修正后 1:1）")
# 若按源文档写法执行次数应为 2*3=6 次 —— 此处输出证明修正有效

① 清洗结果: ['hello world', 'llm ops']
② clean 执行次数: 3 次（元素数 3 个，修正后 1:1）


In [26]:
# ---------- 模式 2：重试装饰器组合（调用不稳定的外部 API） ----------
from functools import wraps

def retry(times):
    """工厂装饰器：参数外置，返回真正的装饰器（三层嵌套）。"""
    def decorator(func):
        @wraps(func)                       # 保留元信息
        def wrapper(*args, **kwargs):
            for i in range(times):
                try:
                    return func(*args, **kwargs)
                except Exception:
                    if i == times - 1:      # 最后一次仍失败：抛出
                        raise
            return None
        return wrapper
    return decorator

def log_call(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        print(f"   [log] {func.__name__} 完成")
        return result
    return wrapper

# 用确定性的「前 N 次失败」模拟不稳定的 LLM API（保证每次执行输出一致）
attempts = {"n": 0}
@log_call
@retry(3)
def call_llm(prompt):
    attempts["n"] += 1
    if attempts["n"] <= 2:          # 前 2 次模拟网络抖动
        raise ConnectionError("模拟网络抖动")
    return f"回复: {prompt[:8]}..."

print("① 结果:", call_llm("介绍一下函数式编程"))
print("② 实际尝试次数:", attempts["n"], "（第 3 次成功）")
print("③ 元信息保留:", call_llm.__name__)

# 一次性失败到底的情况：重试耗尽后向上抛出
attempts2 = {"n": 0}
@retry(3)
def always_fail():
    attempts2["n"] += 1
    raise ValueError("服务持续不可用")
try:
    always_fail()                   # 错误用法演示：重试耗尽必然抛出
except ValueError as e:
    print("④ 重试耗尽触发:", e, "| 尝试次数 =", attempts2["n"])

   [log] call_llm 完成
① 结果: 回复: 介绍一下函数式编...
② 实际尝试次数: 3 （第 3 次成功）
③ 元信息保留: call_llm
④ 重试耗尽触发: 服务持续不可用 | 尝试次数 = 3


In [27]:
# ---------- 模式 3：记忆化缓存（相同请求命中缓存） ----------
from functools import lru_cache

# 模拟一次昂贵的 LLM 调用（真正调用应当是网络请求）
@lru_cache(maxsize=128)            # 设上限，防止缓存无限膨胀
def llm_complete(prompt: str, temperature: float) -> str:
    print(f"   >>> 真实调用 LLM: prompt={prompt!r}, t={temperature}")
    return f"[{prompt} 的回答]"

print("① 第 1 次调用:")
print("   ", llm_complete("什么是纯函数", 0.7))
print("② 第 2 次相同调用（命中缓存，不再打印『真实调用』）:")
print("   ", llm_complete("什么是纯函数", 0.7))
print("③ 换参数（未命中）:")
print("   ", llm_complete("什么是纯函数", 0.2))
print("④ 缓存统计:", llm_complete.cache_info())
print("⑤ 适用场景: 相同 prompt+参数 的重复请求；注意点: 参数必须可哈希、缓存有上限")

① 第 1 次调用:
   >>> 真实调用 LLM: prompt='什么是纯函数', t=0.7
    [什么是纯函数 的回答]
② 第 2 次相同调用（命中缓存，不再打印『真实调用』）:
    [什么是纯函数 的回答]
③ 换参数（未命中）:
   >>> 真实调用 LLM: prompt='什么是纯函数', t=0.2
    [什么是纯函数 的回答]
④ 缓存统计: CacheInfo(hits=1, misses=2, maxsize=128, currsize=2)
⑤ 适用场景: 相同 prompt+参数 的重复请求；注意点: 参数必须可哈希、缓存有上限


In [28]:
# ---------- 模式 4：流式管道（生成器处理大文件/流式响应） ----------
from itertools import islice
from pathlib import Path

def read_lines(path):
    with open(path, encoding="utf-8") as f:
        for line in f:
            yield line.rstrip()          # 逐行惰性读取

def keep_long(lines, n):
    for line in lines:                    # 过滤器也是生成器
        if len(line) >= n:
            yield line

def to_records(lines):
    for line in lines:                    # 变换为结构化记录
        name, score = line.split(",")
        yield {"name": name, "score": int(score)}

big_file = Path("scores_demo.txt")
big_file.write_text("alice,90\nbob,85\ncarol,95\nbo,60\n", encoding="utf-8")

# 管道串联 —— 即使文件有百万行，内存中也只有一行
pipeline = to_records(keep_long(read_lines(big_file), 4))
print("① 管道类型:", type(pipeline).__name__, "（尚未读文件）")
records = list(pipeline)
print("② 结果:", records)
print("③ 平均分:", sum(r["score"] for r in records) / len(records))
print("④ 无限流也能接: 前 3 个自然数的平方 =", list(islice((x * x for x in __import__("itertools").count()), 3)))
big_file.unlink()

① 管道类型: generator （尚未读文件）


② 结果: [{'name': 'alice', 'score': 90}, {'name': 'bob', 'score': 85}, {'name': 'carol', 'score': 95}, {'name': 'bo', 'score': 60}]
③ 平均分: 82.5
④ 无限流也能接: 前 3 个自然数的平方 = [0, 1, 4]


In [29]:
# ---------- 模式 5：不可变更新 + 函数式状态机 ----------
# 5a) 不可变更新：配置热更新不污染原配置（dict 展开式）
def update_config(cfg, **changes):
    return {**cfg, **changes}          # 返回新 dict，原配置不动

base_cfg = {"model": "gpt-4", "temperature": 0.7, "stream": False}
stream_cfg = update_config(base_cfg, stream=True)
print("① 原配置:", base_cfg)
print("② 派生配置:", stream_cfg)

# 5b) 函数式状态机：状态 = 函数，事件 -> 下一状态函数（LLM 请求生命周期）
def idle(event):
    if event == "submit":
        return running
    return idle

def running(event):
    if event == "finish":
        return idle
    if event == "cancel":
        return cancelled
    return running

def cancelled(event):
    if event == "submit":
        return running
    return cancelled

state = idle
trace = []
for event in ["submit", "cancel", "submit", "finish"]:
    state = state(event)
    trace.append(f"{event}->{state.__name__}")
print("③ 状态轨迹:", trace)
print("④ 最终状态:", state.__name__)
print("⑤ 适用: 请求生命周期/审批流；注意: 状态函数必须纯（只看事件决定下一状态）")

① 原配置: {'model': 'gpt-4', 'temperature': 0.7, 'stream': False}
② 派生配置: {'model': 'gpt-4', 'temperature': 0.7, 'stream': True}
③ 状态轨迹: ['submit->running', 'cancel->cancelled', 'submit->running', 'finish->idle']
④ 最终状态: idle
⑤ 适用: 请求生命周期/审批流；注意: 状态函数必须纯（只看事件决定下一状态）


---
# 十六、内置与生态对照

**本章解决什么问题**：函数式相关的工具散落在内置函数、functools、itertools 和第三方库里，一张表看清归属、避免重复造轮子。

**16.1 内置函数 vs 函数式工具**

| 能力 | 内置/标准库 | 说明 | 更地道的替代 |
|---|---|---|---|
| 归约求和 | `reduce(lambda...)` | 通用归约 | `sum(seq)` |
| 归约最大 | `reduce(lambda...)` | 通用归约 | `max(seq)` |
| 全部满足 | — | — | `all(map(f, seq))` |
| 任一满足 | — | — | `any(map(f, seq))` |
| 排序 | `sorted(key=...)` | key 接收函数 | — |
| 映射 | `map(f, seq)` | 惰性 | 推导式 |
| 过滤 | `filter(f, seq)` | 惰性 | 推导式 |
| 反转/枚举/打包 | `reversed / enumerate / zip` | 内置迭代工具 | — |

**16.2 functools / itertools 职责划分**

| 模块 | 解决的问题 | 核心成员 |
|---|---|---|
| `functools` | 「函数」自身的加工 | `partial, lru_cache, cache, reduce, singledispatch, wraps, total_ordering` |
| `itertools` | 「迭代」的组合 | `count, cycle, repeat, chain, zip_longest, product, permutations, combinations, islice, takewhile, dropwhile, filterfalse, groupby, accumulate` |

**16.3 第三方生态（了解即可）**

| 库 | 定位 | 代表能力 |
|---|---|---|
| `toolz` / `cytoolz` | 函数式工具超集 | `pipe, compose, curry, sliding_window, frequencies` |
| `funcy` | 实用函数式糖 | `select, keep, rcompose, silent` |
| `returns` | 函数式错误处理 | `Result/Either` 容器、可组合管线 |
| `pyrsistent` | 持久化数据结构 | `PVector, PMap`（真正的不可变集合） |
| `attrs` / `dataclasses` | 不可变值对象 | `frozen=True` |

---
# 十七、综合实战：LLM API 请求处理管线

**本章解决什么问题**：把前面 12 章的知识点串成一条真实可用的服务端处理链。

**业务场景（本章统一假设）**：LLMOps 平台的 `/v1/complete` 接口要处理一条用户请求：
**清洗 Prompt → 组装不可变配置 → 记忆化查询 → 失败重试 → 流式返回 → 状态跟踪**。

**设计蓝图**

```
用户请求 (prompt, user_id)
    │
    ├─ ① 纯函数清洗管道        —— pipe + 纯函数（第 9 章、模式 1）
    ├─ ② 不可变配置组装        —— dict 展开 / frozen dataclass（第 3 章、模式 5）
    ├─ ③ 记忆化 LLM 调用       —— lru_cache（第 7 章、模式 3）
    ├─ ④ 重试 + 日志装饰器叠加 —— 闭包 + wraps（第 6/7 章、模式 2）
    ├─ ⑤ 流式分块输出          —— 生成器 + islice（第 12 章、模式 4）
    └─ ⑥ 请求生命周期状态机     —— 函数即状态（模式 5b）
```

**设计取舍（重点看）**

| 决策点 | 选择 | 理由 | 代价 |
|---|---|---|---|
| 清洗步骤 | 纯函数 + pipe | 每步可单测、可重排 | 需要约定统一签名 `(str)->str` |
| 配置传递 | 不可变 dict 展开 | 防止并发请求互相污染 | 每次更新都复制（小 dict 可忽略） |
| 缓存层 | `lru_cache` 而非手写 dict | 自带线程安全的 LRU 淘汰 | 参数须可哈希；需设上限防膨胀 |
| 容错 | 装饰器叠加而非 try 块内嵌 | 职责分离，可按需增删 | 三层闭包阅读成本上升 |
| 输出 | 生成器流式 | 大响应不全量驻留内存 | 只能消费一次，重放需重新执行 |
| 生命周期 | 函数式状态机 | 迁移规则一目了然 | 复杂状态图建议改用状态机库 |

In [30]:
# ---------- 综合实战：完整可运行的请求处理管线 ----------
from functools import lru_cache, wraps
from itertools import islice

# ========== ① 纯函数清洗管道（第 9 章 pipe + 模式 1） ==========
def pipe(value, *funcs):
    for f in funcs:
        value = f(value)
    return value

def strip(s):     return s.strip()
def collapse(s):  return " ".join(s.split())

def sanitize_prompt(raw: str) -> str:
    return pipe(raw, strip, collapse)

# ========== ② 不可变配置组装（第 3 章 + 模式 5a） ==========
DEFAULT_CONFIG = {"model": "gpt-4", "temperature": 0.7, "stream": True}

def build_config(**overrides) -> dict:
    return {**DEFAULT_CONFIG, **overrides}     # 新 dict，默认配置永不被污染

# ========== ③ 记忆化 + ④ 重试/日志装饰器（第 6/7 章 + 模式 2/3） ==========
def log_call(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        print(f"   [log] {func.__name__} 完成")
        return result
    return wrapper

def retry(times):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            for i in range(times):
                try:
                    return func(*args, **kwargs)
                except Exception:
                    if i == times - 1:
                        raise
        return wrapper
    return decorator

# 模拟不稳定的 LLM 后端（确定性失败：全局前 1 次调用失败，保证输出可复现）
# 注意装饰器叠加顺序：lru_cache 放最外层 —— 缓存命中时直接返回，跳过日志与重试；
# 且 llm_complete 才能直接访问 cache_info()（若放内层会被外层 wrapper 遮蔽）
_backend = {"calls": 0}

@lru_cache(maxsize=128)
@log_call
@retry(3)
def llm_complete(prompt: str, temperature: float) -> str:
    _backend["calls"] += 1
    if _backend["calls"] == 1:                 # 仅第 1 次真实调用模拟抖动
        raise ConnectionError("模拟网络抖动")
    return f"『{prompt}』的模型回答"

# ========== ⑤ 流式分块输出（第 12 章 + 模式 4） ==========
def stream_chunks(text: str, size: int = 6):
    """把回答按固定块大小流式吐出（模拟 SSE 分块）。"""
    for i in range(0, len(text), size):
        yield text[i:i + size]

# ========== ⑥ 请求生命周期状态机（模式 5b） ==========
def idle(event):
    return processing if event == "request" else idle

def processing(event):
    return done if event == "response" else processing

def done(event):
    return done

# ---------- 组装：一条完整请求的处理链 ----------
state = idle
print("① 初始状态:", state.__name__)

raw_prompt = "   什么是   函数式编程？   "
state = state("request")
print("② 状态迁移:", state.__name__)

clean_prompt = sanitize_prompt(raw_prompt)
cfg = build_config(temperature=0.3)
print("③ 清洗后的 Prompt:", repr(clean_prompt), "| 配置:", cfg)

answer = llm_complete(clean_prompt, cfg["temperature"])   # 首调触发抖动重试
state = state("response")
print("④ 最终状态:", state.__name__)

print("⑤ 流式输出:")
for i, chunk in enumerate(islice(stream_chunks(answer), 4), 1):
    print(f"   chunk{i}: {chunk!r}")

print("⑥ 缓存统计:", llm_complete.cache_info())
print("⑦ 同请求再次调用（命中缓存，0 次网络重试）:", llm_complete(clean_prompt, cfg["temperature"]))
print("⑧ 默认配置未被污染:", DEFAULT_CONFIG)

① 初始状态: idle
② 状态迁移: processing
③ 清洗后的 Prompt: '什么是 函数式编程？' | 配置: {'model': 'gpt-4', 'temperature': 0.3, 'stream': True}
   [log] llm_complete 完成
④ 最终状态: done
⑤ 流式输出:
   chunk1: '『什么是 函'
   chunk2: '数式编程？』'
   chunk3: '的模型回答'
⑥ 缓存统计: CacheInfo(hits=0, misses=1, maxsize=128, currsize=1)
⑦ 同请求再次调用（命中缓存，0 次网络重试）: 『什么是 函数式编程？』的模型回答
⑧ 默认配置未被污染: {'model': 'gpt-4', 'temperature': 0.7, 'stream': True}


---
# 十八、常见陷阱速查表（陷阱 / 症状 / 解法）

**本章解决什么问题**：把散落各章的坑收进一张表；每个陷阱的复现代码都已在前文实际执行过（标注所在章节）。

| # | 陷阱 | 症状（实际观察到的输出/报错） | 解法 | 复现位置 |
|---|---|---|---|---|
| 1 | lambda 延迟绑定 | `[2, 2, 2]` 而非 `[0, 1, 2]` | 默认参数绑定 `lambda i=i: i` | 反例 1 |
| 2 | 可变默认参数 | 第二次调用返回 `[1, 2]`（状态泄漏） | 默认 `None` + 函数体内新建 | 反例 2 |
| 3 | map/filter 返回迭代器 | 第二次 `list()` 得 `[]` | 需要复用就先 `list()` 物化 | 反例 3 |
| 4 | lru_cache 可变参数 | `TypeError: unhashable type` | 改传 tuple 等可哈希类型 | 反例 4 |
| 5 | 闭包漏 nonlocal | `UnboundLocalError` | 内层声明 `nonlocal` | 反例 5 |
| 6 | groupby 不排序 | 同 key 被拆成多段 | 先按分组 key 排序 | 反例 6a |
| 7 | 修改 frozen 实例 | `FrozenInstanceError` | `dataclasses.replace` 新建实例 | 反例 6b |
| 8 | 深递归爆栈 | `RecursionError`（约 1000 层） | 改循环 / 生成器 | 反例 6c |
| 9 | reduce 性能 | 慢于内置 `sum/max` | 简单归约优先用内置函数 | 下个单元 |
| 10 | 过度函数式 | `reduce(lambda acc, x: acc + [x**2], ...)` 难读难调 | 改推导式 | 下个单元 |
| 11 | lambda 套娃不可读 | `lambda x: (lambda y: ...)(2)(3)` | 复杂逻辑一律 `def` | 下个单元 |
| 12 | 副作用混入管线 | 管道中途偷偷改外部状态 | 副作用集中在管线两端 | 下个单元 |

In [31]:
# ---------- 陷阱 9-12 的可运行复现 ----------
import sys
from timeit import timeit
from functools import reduce

# 陷阱 9：reduce 慢于内置 sum（用相对结论而非绝对耗时，保证输出稳定）
nums = list(range(200))
t_reduce = timeit(lambda: reduce(lambda a, b: a + b, nums), number=500)
t_sum = timeit(lambda: sum(nums), number=500)
print("① sum 是否快于 reduce(lambda):", t_sum < t_reduce,
      f"（reduce {t_reduce:.3f}s vs sum {t_sum:.3f}s，结论稳定可复现）")

# 陷阱 10：过度函数式 vs 推导式
result_bad = reduce(lambda acc, x: acc + [x ** 2], filter(lambda x: x % 2 == 0, range(10)), [])
result_good = [x ** 2 for x in range(10) if x % 2 == 0]
print("② 两者结果一致:", result_bad == result_good, "但推导式", "可读性/可调试性显著更优")

# 陷阱 11：lambda 三层套娃（能跑，但没人想维护）
f = lambda x: (lambda y: (lambda z: x + y + z))(2)(3)
print("③ 套娃结果:", f(1), "—— 正确但强烈不推荐，改用 def")

def add_three(x):
    y, z = 2, 3
    return x + y + z
print("④ def 版:", add_three(1), "—— 一眼能懂")

# 陷阱 12：副作用混入管道（管道中段偷偷修改外部状态 -> 结果依赖调用顺序）
side_effect = {"touched": False}
def impure_map(x):
    side_effect["touched"] = True       # 副作用藏在变换函数里
    return x * 2

result = list(map(impure_map, [1, 2, 3]))
print("⑤ 结果:", result, "| 管道是否偷偷改了外部状态:", side_effect["touched"],
      "—— 解法: 变换函数保持纯，副作用集中到管道两端")

print("⑥ 当前递归限制（陷阱 8 附）:", sys.getrecursionlimit())

① sum 是否快于 reduce(lambda): True （reduce 0.002s vs sum 0.000s，结论稳定可复现）
② 两者结果一致: True 但推导式 可读性/可调试性显著更优
③ 套娃结果: 6 —— 正确但强烈不推荐，改用 def
④ def 版: 6 —— 一眼能懂
⑤ 结果: [2, 4, 6] | 管道是否偷偷改了外部状态: True —— 解法: 变换函数保持纯，副作用集中到管道两端
⑥ 当前递归限制（陷阱 8 附）: 1000


---
# 十九、总结

## 19.1 通用代码模板（可直接复制的骨架）

```python
from functools import partial, lru_cache, wraps, reduce
from itertools import islice, chain, groupby

# ① 纯函数清洗管道
def pipe(value, *funcs):
    for f in funcs:
        value = f(value)
    return value

# ② 工厂/闭包：配置化函数
def with_config(**defaults):
    def make(**overrides):
        return {**defaults, **overrides}      # 不可变更新
    return make

# ③ 记忆化（参数必须可哈希）
@lru_cache(maxsize=128)
def expensive(key: str) -> str: ...

# ④ 重试装饰器（三层闭包）
def retry(times):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            for i in range(times):
                try:
                    return func(*args, **kwargs)
                except Exception:
                    if i == times - 1:
                        raise
        return wrapper
    return decorator

# ⑤ 流式生成器管道
def stream(source):
    for item in source:        # source 本身也是生成器
        if keep(item):
            yield transform(item)

# ⑥ 函数式状态机
def state_a(event):
    return state_b if event == "go" else state_a
```

## 19.2 记忆口诀

1. **一等函数是地基**：能赋值、能传参、能返回、能进容器。
2. **纯函数 + 不可变**：同进同出不碰外，副作用集中两端管。
3. **简单变换用推导式**：map/filter 带 lambda 时，先想想推导式。
4. **归约优先用内置**：sum/max/any/all 快过 reduce。
5. **工具箱别造轮子**：functools 管函数加工，itertools 管迭代组合。
6. **迭代器是一次性的**：要复用先物化，无限流必须 islice。
7. **lambda 只写一行**：套娃即重构，闭包记 nonlocal。

## 19.3 自测练习（先自己写，再看参考实现）

| # | 题目 | 考察点 |
|---|---|---|
| 1 | 写一个 `compose(f, g, h)`，验证 `compose(f,g,h)(x)` 等价于 `f(g(h(x)))` | 函数组合 + 执行顺序 |
| 2 | 用纯函数实现 `merge_config(base, overrides)`，并证明原 dict 未被修改 | 不可变更新 |
| 3 | 给递归版 `fib` 加缓存，并对比加缓存前后的「函数体执行次数」 | 记忆化 |
| 4 | 写生成器 `read_batches(path, n)`，每次 yield n 行（文件自备） | 惰性求值 |
| 5 | 解释 `[lambda: i for i in range(3)]` 为何输出 `[2,2,2]`，并给出两种修复 | 延迟绑定 |

In [32]:
# ---------- 自测参考实现（先自己写，再对照） ----------
# 题目 1：compose(f, g, h) 的执行顺序验证
def compose(*funcs):
    def composed(x):
        for f in reversed(funcs):
            x = f(x)
        return x
    return composed

f1, f2, f3 = (lambda x: x + 1), (lambda x: x * 2), (lambda x: x - 3)
print("① compose 版:", compose(f1, f2, f3)(10), "| 嵌套版:", f1(f2(f3(10))), "（相等 =",
      compose(f1, f2, f3)(10) == f1(f2(f3(10))), "）")

# 题目 2：纯函数配置合并
def merge_config(base, overrides):
    return {**base, **overrides}

base_cfg = {"model": "gpt-4", "temperature": 0.7}
merged = merge_config(base_cfg, {"temperature": 0.2})
print("② 合并结果:", merged, "| 原 base 是否被改:", base_cfg != merged)

# 题目 3：fib 记忆化前后对比
calls = {"n": 0}

def fib_plain(n):
    calls["n"] += 1
    return n if n < 2 else fib_plain(n - 1) + fib_plain(n - 2)
fib_plain(20)
print("③ 无缓存执行次数:", calls["n"], "次")

from functools import lru_cache
calls2 = {"n": 0}

@lru_cache(maxsize=None)
def fib_cached(n):
    calls2["n"] += 1
    return n if n < 2 else fib_cached(n - 1) + fib_cached(n - 2)
fib_cached(20)
print("   有缓存执行次数:", calls2["n"], "次（20 层只算 21 次）")

# 题目 4：批量读文件生成器
from pathlib import Path

def read_batches(path, n):
    batch = []
    with open(path, encoding="utf-8") as fh:
        for line in fh:
            batch.append(line.rstrip())
            if len(batch) == n:
                yield batch          # 攒满一批就交出去
                batch = []
    if batch:
        yield batch                  # 别忘了尾部不满一批的

tmp = Path("batch_demo.txt")
tmp.write_text("a\nb\nc\nd\ne\n", encoding="utf-8")
print("④ 分批结果:", list(read_batches(tmp, 2)))
tmp.unlink()

# 题目 5：延迟绑定的原因与两种修复
funcs_bad = [lambda: i for i in range(3)]            # 共享循环变量 i（惰性求值）
funcs_fix1 = [lambda i=i: i for i in range(3)]        # 修复 1：默认参数立即绑定
funcs_fix2 = [ (lambda v: lambda: v)(i) for i in range(3)]  # 修复 2：再包一层闭包
print("⑤ 错误:", [f() for f in funcs_bad],
      "修复1:", [f() for f in funcs_fix1],
      "修复2:", [f() for f in funcs_fix2])

① compose 版: 15 | 嵌套版: 15 （相等 = True ）
② 合并结果: {'model': 'gpt-4', 'temperature': 0.2} | 原 base 是否被改: True
③ 无缓存执行次数: 21891 次
   有缓存执行次数: 21 次（20 层只算 21 次）
④ 分批结果: [['a', 'b'], ['c', 'd'], ['e']]


⑤ 错误: [2, 2, 2] 修复1: [0, 1, 2] 修复2: [0, 1, 2]


---
# 二十、收尾：一图流回顾

> **Python 的函数式编程不是要取代命令式，而是提供一种「用函数组合表达计算」的思路；
> 推导式、生成器、functools、itertools 是它的四大支柱。**

| 核心概念 | 一句话 | 关键 API |
|---|---|---|
| 一等函数 | 函数可赋值、传递、返回 | — |
| 纯函数 | 无副作用、同进同出 | `lru_cache` 的前提 |
| 不可变性 | 不改原数据、返回新数据 | `{**d}`, `replace()` |
| 高阶函数 | 接收/返回函数 | `map/filter/reduce` |
| 闭包 | 函数记住定义环境 | `nonlocal` |
| 柯里化 | 多参拆成单参链 | 嵌套 lambda |
| 函数组合 | 小函数串成流水线 | `compose/pipe` |
| 惰性求值 | 按需计算 | `yield/islice` |

**建议的学习闭环**：把第十七章的综合实战管线抄下来 → 删掉自己重写 → 跑通 → 再扩展一个你业务里真实的环节（如加一个限流装饰器）。

（完）